## SHAP & business cutoff

Explains the final LightGBM from `03_lightgbm` and turns its scores into an approval decision.
The model is loaded from MLflow, not retrained. Same data preparation and train/test split as `02_baseline` and `03_lightgbm`.
SHAP shows which features drive the risk. The cutoff analysis translates the ranking into approval rate and defaults avoided.

**1 - Install SHAP and LightGBM**

In [0]:
%pip install shap lightgbm
dbutils.library.restartPython()

**2 - Imports and constants**

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import shap
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

PATH = "/Volumes/workspace/risco_credito/raw/cs-training.csv"
TARGET = "SeriousDlqin2yrs"
PAST_DUE = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]
RANDOM_STATE = 42

RUN_ID = "bff55fc3fdbb441bad37b58239f23ae8"  # lightgbm_final from 03_lightgbm
EXPECTED_TEST_PR_AUC = 0.4094

**3 - Read CSV, split and load the model from MLflow**

In [0]:
df = pd.read_csv(PATH, index_col=0)

df["past_due_sentinel"] = df[PAST_DUE].isin([96, 98]).any(axis=1).astype(int)
df["income_missing"] = df["MonthlyIncome"].isna().astype(int)
df["dependents_missing"] = df["NumberOfDependents"].isna().astype(int)
df.loc[df["age"] == 0, "age"] = np.nan
df[PAST_DUE] = df[PAST_DUE].where(~df[PAST_DUE].isin([96, 98]))

X = df.drop(columns=TARGET)
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

model = mlflow.lightgbm.load_model(f"runs:/{RUN_ID}/model")
proba_test = model.predict_proba(X_test)[:, 1]

pr_auc = average_precision_score(y_test, proba_test)
print(f"roc_auc {roc_auc_score(y_test, proba_test):.4f} | pr_auc {pr_auc:.4f}")

# Same model, same data -> same score. If not, something changed upstream.
assert abs(pr_auc - EXPECTED_TEST_PR_AUC) < 1e-4, "Loaded model does not reproduce 03_lightgbm"

**4 - Compute SHAP values and check additivity**

In [0]:
explainer = shap.TreeExplainer(model)
sv = explainer(X_test)

# Some shap versions return one column per class; keep class 1 (default)
if sv.values.ndim == 3:
    sv = sv[:, :, 1]

print(f"shap values shape: {sv.values.shape}")

# Additivity: base value + sum of contributions = model raw score (log-odds)
raw = model.predict(X_test, raw_score=True)
reconstructed = sv.base_values + sv.values.sum(axis=1)
print(f"max additivity error: {np.abs(raw - reconstructed).max():.2e}")

base = np.atleast_1d(sv.base_values)[0]
print(f"base value: {base:.3f} log-odds -> {1 / (1 + np.exp(-base)):.2%} | train default rate {y_train.mean():.2%}")

**5 - Global importance and beeswarm**

In [0]:
import os

importance = (
    pd.Series(np.abs(sv.values).mean(axis=0), index=X_test.columns)
    .sort_values(ascending=False)
    .rename("mean_abs_shap")
)
display(importance.round(3).to_frame())

shap.plots.bar(sv, max_display=13, show=False)
plt.title("Global importance (mean |SHAP|, log-odds)")
plt.tight_layout()
plt.show()

shap.plots.beeswarm(sv, max_display=13, show=False)
plt.title("SHAP values per borrower (test set)")
plt.tight_layout()
os.makedirs("../images", exist_ok=True)
plt.savefig("../images/shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()

**6 - Predicted vs. actual default rate for missing-value groups**

In [0]:
check = pd.DataFrame({
    "actual": y_test.values,
    "predicted": proba_test,
    "past_due_sentinel": X_test["past_due_sentinel"].values,
    "income_missing": X_test["income_missing"].values,
})

for flag in ["past_due_sentinel", "income_missing"]:
    summary = check.groupby(flag).agg(
        n=("actual", "size"),
        actual=("actual", "mean"),
        predicted=("predicted", "mean"),
    )
    print(f"\n{flag}")
    print(summary.round(3))

**7 - DebtRatio dependence plot, split by income_missing**

In [0]:
dr = X_test["DebtRatio"].values
dr_shap = sv[:, "DebtRatio"].values
missing = X_test["income_missing"].values == 1

print(f"median DebtRatio - income reported: {np.median(dr[~missing]):.2f} | income missing: {np.median(dr[missing]):.0f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

axes[0].scatter(dr[~missing], dr_shap[~missing], s=4, alpha=0.3)
axes[0].set_xlim(0, 3)  # most borrowers are below 1
axes[0].set_title("Income reported: DebtRatio is a ratio")
axes[0].set_xlabel("DebtRatio")
axes[0].set_ylabel("SHAP value (log-odds)")

axes[1].scatter(dr[missing], dr_shap[missing], s=4, alpha=0.3, color="tab:orange")
axes[1].set_xscale("symlog")  # values range from 0 to thousands
axes[1].set_title("Income missing: DebtRatio is an absolute amount")
axes[1].set_xlabel("DebtRatio (symlog scale)")

for ax in axes:
    ax.axhline(0, color="gray", linewidth=0.8)

plt.tight_layout()
plt.show()

**Reading the plot:**
* **The same column holds two different things.** When income is reported, `DebtRatio` is a ratio (monthly debt payments / income). When income is missing, there is nothing to divide by, so the value is the debt amount itself, often in the thousands.
* **Income reported (left):** risk is lowest between about 10% and 40% of income committed. It rises sharply above 50% and stays high past 100%. Borrowers with a ratio of exactly 0 also show slightly higher risk, possibly because they have little credit history.
* **Income missing (right):** the pattern is different. Small amounts slightly increase risk, while amounts between about 1,000 and 5,000 reduce it. One possible reason is that large debts, such as mortgages, mean a lender has already approved the borrower. This is a hypothesis, not something the data proves.
* **Why the logistic regression got a negative coefficient:** it fits one straight line per feature. The borrowers with missing income have very high `DebtRatio` values and a below-average default rate (5.8%), which pulled the line down. LightGBM splits the data into ranges, so it learns both patterns separately.
* **Possible improvement:** split the column into `DebtRatio` (NaN when income is missing) and a new `DebtAmount` feature. Not done here to keep the scope fixed.

**8 - Out-of-fold scores on train and cutoff table**

- Choosing an approval cutoff is a decision, so it should not be made on the test set. Instead, each borrower in train gets an out-of-fold (OOF) score: the train set is split into 5 folds, and each fold is scored by a model trained on the other 4.
- `clone(model)` copies the MLflow model's hyperparameters (119 trees, same settings) without its training, so the OOF scores come from the same model setup.
- The OOF PR-AUC should be close to the CV result from `03_lightgbm` (~0.40).
- For each approval rate from 50% to 95%, the table shows:
  - `threshold`: the score cutoff. Borrowers above it are rejected.
  - `default_rate_approved`: default rate among approved borrowers.
  - `defaults_avoided`: share of all defaults that fall in the rejected group.
- Reference: approving everyone gives a 6.68% default rate and avoids no defaults.
- The cutoff is chosen on these OOF scores and then applied once to the test set.

In [0]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_val_predict

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
proba_oof = cross_val_predict(clone(model), X_train, y_train, cv=cv, method="predict_proba")[:, 1]
print(f"OOF pr_auc: {average_precision_score(y_train, proba_oof):.4f}")

def cutoff_table(y, proba, approval_rates):
    y = np.asarray(y)
    rows = []
    for rate in approval_rates:
        threshold = np.quantile(proba, rate)
        approved = proba <= threshold
        rows.append({
            "approval_rate": approved.mean(),
            "threshold": threshold,
            "default_rate_approved": y[approved].mean(),
            "defaults_avoided": y[~approved].sum() / y.sum(),
        })
    return pd.DataFrame(rows)

table_oof = cutoff_table(y_train, proba_oof, np.arange(0.50, 0.96, 0.05))
display(table_oof.round(3))

**9 - Apply the 80% approval cutoff once to the test set**

In [0]:
THRESHOLD = np.quantile(proba_oof, 0.80)  # chosen on OOF train scores

approved = proba_test <= THRESHOLD
y_arr = y_test.values
print(f"threshold: {THRESHOLD:.3f}")
print(f"approval rate: {approved.mean():.1%}")
print(f"default rate approved: {y_arr[approved].mean():.1%} (everyone: {y_arr.mean():.1%})")
print(f"defaults avoided: {y_arr[~approved].sum() / y_arr.sum():.1%} ({y_arr[~approved].sum()} of {y_arr.sum()})")

- The cutoff is fixed at the 80th percentile of the OOF train scores. Up to 80% approval, each extra 5 points of approval costs about 6 points of defaults avoided. Above 80%, the cost grows quickly (7, 11, then 19 points).
- The fixed cutoff is applied once to the test set to report the final numbers.

**10 - Findings**

* **Main risk drivers:** credit card utilization, past-due counts and age. Income and dependents have a small effect. The logistic regression in `02_baseline` found the same drivers.
* **Missing-value flags have SHAP = 0 in LightGBM.** The same information is in the NaN values of the original columns, so the flags are never used. LightGBM predicted the sentinel group at 53.9% (54.5% actual) using only the NaN values. In the logistic regression, the flags were needed because imputation removed the NaN values.
* **Missing income is not a risk signal here.** Borrowers without reported income default less (5.8%) than those with income (6.9%).
* **`DebtRatio` hides two variables.** With income reported, risk is lowest at 10–40% of income committed and rises above 50%. With income missing, the column is a debt amount and follows a different pattern. This explains the negative coefficient in `02_baseline`.
* **Base value of 2.9% vs. 6.7% default rate:** SHAP works in log-odds. Most borrowers have very low risk and a small group has very high risk, so the mean in log-odds converts to a lower probability.
* **Business impact (test set):** approving 80% of applicants gives a default rate of <X>% among approved borrowers (6.7% if everyone is approved) and avoids <Y>% of all defaults.